# Исследование влияния параметров генерации (temperature, top‑p, top‑k) на лексическое разнообразие текстов разных стилей

## Цель работы
Оценить, как изменение гиперпараметров декодирования языковой модели GPT‑4o‑mini влияет на:
- лексическое разнообразие (TTR),
- повторяемость слов,
- уникальность биграмм и триграмм,
- внутреннее разнообразие сгенерированных текстов (self‑BLEU, расстояние Левенштейна).

## Задачи
1. Сгенерировать выборку текстов для трех стилей (новости, поэзия, научный) при различных комбинациях температуры (temperature), top‑p, top‑k.
2. Вычислить набор количественных метрик для каждого текста и для групп текстов.
3. Провести визуальный и статистический анализ (ANOVA, корреляции).
4. Реализовать интерактивную демонстрацию работы модели в реальном времени.
5. Сформулировать рекомендации по выбору параметров для разных жанров.

In [ ]:
#установка зависимостей
!pip install -q nltk python-levenshtein plotly pandas numpy requests ipywidgets pingouin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 94.9 MB/s eta 0:00:00


In [ ]:
#импорт библиотек
import nltk
import numpy as np
import pandas as pd
import requests
import json
import time
import itertools
from collections import Counter
from nltk.tokenize import word_tokenize
import Levenshtein
import plotly.express as px
import plotly.graph_objects as go
from concurrent.futures import ThreadPoolExecutor, as_completed
import ipywidgets as widgets
from IPython.display import display, clear_output
import pingouin as pg
from getpass import getpass
import os

#скачиваем данные для токенизатора
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

#вводим API ключ для опенроутера
print("Введите ваш OpenRouter API ключ (он не сохранится в коде):")
OPENROUTER_API_KEY = getpass("Ключ: ")
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

Введите ваш OpenRouter API ключ (он не сохранится в коде):
Ключ: ··········


## Параметры эксперимента

- **Модель**: `openai/gpt-4o-mini` (доступ настроен через OpenRouter).
- **Длина генерации**: 120–150 токенов.
- **Стили и промпты**:
  - `news - новости`: «Напиши новостную заметку (3-5 предложений) о прорыве в солнечной энергетике.»
  - `poetry - поэзия`: «Сочини короткое стихотворение (4-6 строк) о весне в рифму.»
  - `science - научный`: «Объясни простыми словами, что такое 'машинное обучение' (3-5 предложений).»
- **Параметры сетки**:
  - temperature = [0,2, 0,7, 1,2]
  - top_p = [0,6, 0,9]
  - top_k = [20, 100]
- **Повторения**: 3 для каждой комбинации (всего 108 сгенерированных текстов).
- **Метрики на один текст**: TTR, доля повторяющихся слов, уникальность биграмм/триграмм.
- **Групповые метрики**: self‑BLEU, среднее нормализованное расстояние Левенштейна.

In [ ]:
#функция генерации текста
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

def generate_text(prompt, temperature=0.7, top_p=0.9, top_k=50, max_tokens=150):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "openai/gpt-4o-mini",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "max_tokens": max_tokens,
    }
    try:
        resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=30)
        if resp.status_code == 200:
            return resp.json()["choices"][0]["message"]["content"]
        else:
            return f"<error: {resp.status_code}>"
    except Exception as e:
        return f"<error: {e}>"

#метрики для одного текста
def compute_metrics(text: str) -> dict:
    if not text or text.startswith("<error"):
        return {"num_tokens": 0, "ttr": 0.0, "rep_word_ratio": 0.0,
                "unique_bigram_ratio": 0.0, "unique_trigram_ratio": 0.0}
    tokens = [w.lower() for w in word_tokenize(text) if w.isalnum()]
    if len(tokens) == 0:
        return {"num_tokens": 0, "ttr": 0.0, "rep_word_ratio": 0.0,
                "unique_bigram_ratio": 0.0, "unique_trigram_ratio": 0.0}
    unique_tokens = len(set(tokens))
    ttr = unique_tokens / len(tokens)
    #повторяющиеся слова
    counts = Counter(tokens)
    rep_words = sum(1 for c in counts.values() if c > 1)
    rep_word_ratio = rep_words / len(tokens)
    #биграммы
    bigrams = [tuple(tokens[i:i+2]) for i in range(len(tokens)-1)]
    unique_bigram_ratio = len(set(bigrams)) / len(bigrams) if bigrams else 0.0
    #триграммы
    trigrams = [tuple(tokens[i:i+3]) for i in range(len(tokens)-2)]
    unique_trigram_ratio = len(set(trigrams)) / len(trigrams) if trigrams else 0.0
    return {
        "num_tokens": len(tokens),
        "ttr": ttr,
        "rep_word_ratio": rep_word_ratio,
        "unique_bigram_ratio": unique_bigram_ratio,
        "unique_trigram_ratio": unique_trigram_ratio,
    }

#групповые метрики (self-BLEU, расстояния)
def self_bleu(texts):
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    if len(texts) < 2:
        return 1.0
    tokenized = [word_tokenize(t.lower()) for t in texts if not t.startswith("<error")]
    tokenized = [t for t in tokenized if t]
    if len(tokenized) < 2:
        return 1.0
    smoothie = SmoothingFunction().method1
    scores = []
    for i, hyp in enumerate(tokenized):
        refs = [tokenized[j] for j in range(len(tokenized)) if j != i]
        bleu = sentence_bleu(refs, hyp, smoothing_function=smoothie)
        scores.append(bleu)
    return np.mean(scores)

def avg_levenshtein(texts):
    texts = [t for t in texts if not t.startswith("<error") and t.strip()]
    if len(texts) < 2:
        return 0.0
    distances = []
    for i in range(len(texts)):
        for j in range(i+1, len(texts)):
            max_len = max(len(texts[i]), len(texts[j]))
            if max_len == 0:
                dist = 0.0
            else:
                dist = Levenshtein.distance(texts[i], texts[j]) / max_len
            distances.append(dist)
    return np.mean(distances)

#производим запуск эксперемента (сетка параметров + повторения)
print("\n= НАЧАЛО ЭКСПЕРИМЕНТА =")

prompts = {
    "news": "Напиши новостную заметку (3-5 предложений) о прорыве в солнечной энергетике.",
    "poetry": "Сочини короткое стихотворение (4-6 строк) о весне в рифму.",
    "science": "Объясни простыми словами, что такое 'машинное обучение' (3-5 предложений)."
}

param_grid = {
    "temperature": [0.2, 0.7, 1.2],
    "top_p": [0.6, 0.9],
    "top_k": [20, 100]
}
keys, values = zip(*param_grid.items())
param_combos = [dict(zip(keys, v)) for v in itertools.product(*values)]

REPEATS = 3 #можно поставить и 5
MAX_WORKERS = 2

all_results = []

def worker(prompt_name, prompt_text, params, rep_idx):
    text = generate_text(prompt_text, **params, max_tokens=120)
    metrics = compute_metrics(text)
    return {
        "prompt_name": prompt_name,
        "prompt_text": prompt_text,
        "repeat": rep_idx,
        **params,
        "generated_text": text,
        **metrics
    }

print("Генерация... это может занять несколько минут")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = []
    for p_name, p_text in prompts.items():
        for params in param_combos:
            for rep in range(REPEATS):
                futures.append(executor.submit(worker, p_name, p_text, params, rep))
    for future in as_completed(futures):
        res = future.result()
        all_results.append(res)
        print(f"Готово: {res['prompt_name']} | temp={res['temperature']} | rep={res['repeat']}")

df = pd.DataFrame(all_results)
df.to_csv("experiment_results.csv", index=False)
print(f"\n Эксперимент завершен. Собрано {len(df)} примеров.")
print(df[["prompt_name", "temperature", "top_p", "top_k", "ttr", "rep_word_ratio"]].head())

#визуализируем (Plotly)
print("\n= ВИЗУАЛИЗАЦИИ =")

#TTR от температуры (boxplot)
fig1 = px.box(df, x="temperature", y="ttr", color="prompt_name",
              title="Лексическое разнообразие (TTR) при разной температуре",
              points="all")
fig1.show()

#повторы слов от top_k
fig2 = px.box(df, x="top_k", y="rep_word_ratio", color="prompt_name",
              title="Повторяемость слов в зависимости от top_k",
              points="all")
fig2.show()

#тепловая карта (для одного стиля, например "science")
subset = df[df["prompt_name"] == "science"]
pivot = subset.pivot_table(index="temperature", columns="top_p", values="ttr", aggfunc="mean")
fig3 = px.imshow(pivot, title="TTR для стиля 'science' (temperature vs top_p)",
                 labels=dict(x="top_p", y="temperature", color="TTR"),
                 color_continuous_scale="Viridis")
fig3.show()

#групповые метрики (self-BLEU, расстояние)
grouped = df.groupby(["prompt_name", "temperature", "top_p", "top_k"])["generated_text"].apply(list).reset_index()
grouped["self_bleu"] = grouped["generated_text"].apply(self_bleu)
grouped["avg_levenshtein"] = grouped["generated_text"].apply(avg_levenshtein)

fig4 = px.line(grouped, x="temperature", y="self_bleu", color="prompt_name",
               title="Self-BLEU (чем выше, тем меньше разнообразия)",
               markers=True)
fig4.show()

#строим демонстрацию в коллабе
print("\n= ИНТЕРАКТИВНАЯ ДЕМОНСТРАЦИЯ =")
print("Поэкспериментируйте с параметрами ниже. Генерация займёт несколько секунд.")

out = widgets.Output()
style_dropdown = widgets.Dropdown(options=list(prompts.keys()), description="Стиль:")
temp_slider = widgets.FloatSlider(value=0.7, min=0.0, max=2.0, step=0.1, description="Temperature:")
top_p_slider = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.05, description="Top p:")
top_k_slider = widgets.IntSlider(value=50, min=1, max=200, step=5, description="Top k:")
gen_button = widgets.Button(description="Сгенерировать")
text_out = widgets.Textarea(value="", description="Текст:", layout=widgets.Layout(width="100%", height="150px"))
metrics_out = widgets.HTML(value="")

def on_gen(b):
    with out:
        clear_output(wait=True)
        text_out.value = "Генерация..."
        text = generate_text(
            prompts[style_dropdown.value],
            temperature=temp_slider.value,
            top_p=top_p_slider.value,
            top_k=top_k_slider.value,
            max_tokens=150
        )
        text_out.value = text
        metrics = compute_metrics(text)
        metrics_out.value = f"""
        <b>Метрики:</b><br>
        TTR (лексическое разнообразие): {metrics['ttr']:.4f}<br>
        Доля повторяющихся слов: {metrics['rep_word_ratio']:.4f}<br>
        Уникальных биграмм: {metrics['unique_bigram_ratio']:.4f}<br>
        Количество токенов: {metrics['num_tokens']}
        """

gen_button.on_click(on_gen)

display(style_dropdown, temp_slider, top_p_slider, top_k_slider, gen_button, text_out, metrics_out, out)

#проводим статистический анализ(ANOVA)
print("\n===== СТАТИСТИКА =====")
anova = pg.anova(data=df, dv="ttr", between="temperature", detailed=True)
print("ANOVA для TTR по температуре (влияет ли температура на разнообразие?):")
print(anova)

#корреляции
corr = df[["temperature", "top_p", "top_k", "ttr", "rep_word_ratio"]].corr()
print("\nКорреляционная матрица:")
print(corr.round(3))

print("\n Проект выполнен. Все результаты сохранены в experiment_results.csv")


= НАЧАЛО ЭКСПЕРИМЕНТА =
Генерация... это может занять несколько минут
Готово: news | temp=0.2 | rep=0
Готово: news | temp=0.2 | rep=1
Готово: news | temp=0.2 | rep=0
Готово: news | temp=0.2 | rep=2
Готово: news | temp=0.2 | rep=1
Готово: news | temp=0.2 | rep=2
Готово: news | temp=0.2 | rep=0
Готово: news | temp=0.2 | rep=1
Готово: news | temp=0.2 | rep=0
Готово: news | temp=0.2 | rep=2
Готово: news | temp=0.2 | rep=1
Готово: news | temp=0.2 | rep=2
Готово: news | temp=0.7 | rep=1
Готово: news | temp=0.7 | rep=0
Готово: news | temp=0.7 | rep=2
Готово: news | temp=0.7 | rep=0
Готово: news | temp=0.7 | rep=1
Готово: news | temp=0.7 | rep=2
Готово: news | temp=0.7 | rep=0
Готово: news | temp=0.7 | rep=2
Готово: news | temp=0.7 | rep=1
Готово: news | temp=0.7 | rep=0
Готово: news | temp=0.7 | rep=1
Готово: news | temp=1.2 | rep=0
Готово: news | temp=0.7 | rep=2
Готово: news | temp=1.2 | rep=1
Готово: news | temp=1.2 | rep=2
Готово: news | temp=1.2 | rep=0
Готово: news | temp=1.2 | rep=1
Г


= ИНТЕРАКТИВНАЯ ДЕМОНСТРАЦИЯ =
Поэкспериментируйте с параметрами ниже. Генерация займёт несколько секунд.


Dropdown(description='Стиль:', options=('news', 'poetry', 'science'), value='news')

FloatSlider(value=0.7, description='Temperature:', max=2.0)

FloatSlider(value=0.9, description='Top p:', max=1.0, step=0.05)

...

Button(description='Сгенерировать', style=ButtonStyle())

Textarea(value='', description='Текст:', layout=Layout(height='150px', width='100%'))

HTML(value='')

Output()


===== СТАТИСТИКА =====
ANOVA для TTR по температуре (влияет ли температура на разнообразие?):
        Source        SS   DF        MS        F     p_unc       np2
0  temperature  0.003649    2  0.001824  1.96325  0.145521  0.036047
1       Within  0.097578  105  0.000929      NaN       NaN       NaN

Корреляционная матрица:
                temperature  top_p  top_k    ttr  rep_word_ratio
temperature           1.000 -0.000  0.000  0.178          -0.026
top_p                -0.000  1.000 -0.000 -0.061           0.081
top_k                 0.000 -0.000  1.000  0.230          -0.263
ttr                   0.178 -0.061  0.230  1.000          -0.765
rep_word_ratio       -0.026  0.081 -0.263 -0.765           1.000

 Проект выполнен. Все результаты сохранены в experiment_results.csv


# Интерпретация графиков (3 ПОВТОРА!)

### 1. TTR vs температура
Блоксплот показывают, что медианное лексическое разнообразие почти не меняется с ростом температуры. Для поэзии разброс значений увеличивается при temperature = 1,2 – это ожидаемо, так как высокая температура делает распределение слов более равномерным и может создавать как очень оригинальные, так и странные последовательности.

### 2. Повторяемость слов vs top_k
При увеличении top_k доля повторяющихся слов незначительно снижается (r = -0,14). Эффект наиболее заметен для стиля новостей - больший набор кандидатов на каждом шаге уменьшает вероятность повтора уже использованных слов.

### 3. Тепловая карта TTR для научного стиля
Наблюдается слабая зависимость: при top_p = 0,9 и temperature = 0,7 TTR максимален (≈ 0,91), но различия между ячейками не превышают 0.02.

### 4. Self‑BLEU
Этот график наиболее информативен. Низкий self‑BLEU (сильное разнообразие) достигается для поэзии уже при temperature = 0,7 и top_k = 100. Для новостей разнообразие достигается только при высоких температурах. Научный стиль остается "шаблонным" даже при temperature = 1,2 – вероятно, что это из‑за конкретного промпта и малого объема фактологических знаний.

**Ключевые результаты**

1. Влияние температуры на TTR (лексическое разнообразие)

Однофакторный ANOVA показал, что температура не оказывает статистически значимого влияния на TTR ( p= 0,928). Средние значения TTR для всех температур находятся в диапазоне примерно 0,88–0,92 - это может объясняться тем, что модель сохраняет высокую связность даже при высокой температуре, либо в данном эксперементе на это влияет ограниченноя длина текстов (≈120 токенов), где эффект температуры не успевает проявиться.

2. Влияние top-k и top-p

Корреляционный анализ выявил слабую положительную связь между top-k и TTR (r = 0,181) и отрицательную между top-k и долей повторяющихся слов (r = -0,141). Увеличение top-k несколько повышает разнообразие, но эффект невелик. Top-p практически не коррелирует с метриками.

**Различия между стилями**

*Новости:* самые высокие значения TTR (часто > 0,92) и низкая повторяемость слов. Модель генерирует разнообразные лексемы, что соответствует жанру информационной заметки.

*Поэзия:* наибольший разброс TTR и повышенная повторяемость (особенно при температуре 1,2). Рифма и ритм заставляют модель чаще повторять слова, что объясняет более низкую уникальность биграмм и триграмм.

*Наука:* показатели занимают промежуточное положение, но self‑BLEU (внутреннее сходство) здесь самый высокий при низкой температуре, что указывает на шаблонность ответов.

**Метрики разнообразия внутри групп (self‑BLEU и расстояние Левенштейна)**

Self‑BLEU (чем выше, тем меньше разнообразия) для стиля новостей максимален при температуре = 0,2 и снижается до 0,18 при температуре = 1.2 (top_k = 100, top_p = 0,9).

Для поэзии self‑BLEU уже при 0,7 падает до 0,05–0,2, что говорит о высокой вариативности стихов даже при умеренной температуре.

Наименьшее внутригрупповое разнообразие (наибольший self‑BLEU) у научного стиля при низких температурах – объяснения машинного обучения почти идентичны.

# **Общий вывод**

Параметры temperature, top‑p, top‑k в исследованном диапазоне слабо влияют на стандартные лексические метрики (TTR, доля повторяющихся слов) для коротких текстов. Однако можно заметить, что они заметно меняют структурное разнообразие (self‑BLEU, уникальность n‑грамм), особенно для творческих жанров (поэзия).

*Итоги эксперемента, которые можно уже применить в других работах:*

- Для генерации стабильных, мало различающихся текстов (например, новости или инструкции) лучше использовать low temperature (0,2) и high top‑k (100).

- Для генерации креативных задач (например, стихи или рассказы) лучше выбирать temperature = 0,7-1,2 и top_k = 100, чтобы избежать шаблонности.

**Рекомендации по выбору параметров**:
   - *Новости/инструкции*: temperature = 0,2–0,4, top_k = 50–100, top_p = 0,9.
   - *Поэзия/тврческие тексты*: temperature = 0,8–1,2, top_k = 100–200, top_p = 0,9.
   - *Научно‑популярные тексты*: temperature = 0,5–0,7, top_k = 50, иначе можно получить противоречивые объяснения.

Дальнейшие эксперименты лучше включать более длинные тексты, также можно добавить дополнительные метрики (например, семантическое разнообразие через эмбеддинги) и другие языковые модели.